# HW#5 -- Part I: Summarization using Encoder-Decoder Models

**Name:** [Your Name]
**Course:** CSC 583 -- Natural Language Processing, Fall 2025
**Assignment:** HW#5, Part I
**Collaborators:** [List any collaborators, or "None"]

## 0. Setup (Google Colab)

This part is done for you -- mount your Drive, `cd` into your working folder, install the
libraries you'll need, and import them.

In [ ]:
## Mount Google Drive (so results/files can be saved/loaded persistently)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Change the working directory to this assignment's folder.
import os

thisdir = "/content/drive/My Drive/CSC583_Fall2026/HW5"   # <-- update to YOUR folder if different
os.chdir(thisdir)
!pwd

In [ ]:
!pip install -q -U transformers tokenizers accelerate huggingface_hub datasets evaluate sentencepiece rouge_score bert_score

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from evaluate import load as load_metric

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Load the test set -- `RealTimeData/bbc_news_alltime`

Sample 300 random examples from the **most recent complete month** available on the HF hub
(check the dataset's "Subsets" list on its HF page). Use the `'content'` column as the
original text and `'description'` as the reference summary -- filter out any examples with
no description first.

In [ ]:
# TODO (Section 1): Load and prepare the BBC News test set.
#
#   1. Set MONTH to the most recent COMPLETE month available on the HF hub right now,
#      e.g. "2025-12" (see the dataset's "Subsets" list on its HF page).
#   2. Load the "train" split for that month/config with load_dataset(...).
#   3. Filter out examples where "description" is None or empty.
#   4. Shuffle with seed=42 and select 300 examples -- this is your test set.
#
# Required variable when you're done:
#   news  -- a Dataset of 300 examples with "content" and "description" columns

MONTH = None   # TODO: set this
news = None    # TODO: build this

raise NotImplementedError("TODO: load and prepare the BBC News test set (Section 1)")

## 2. Load the two models + tokenizers

Both `google-t5/t5-small` and `Falconsai/text_summarization` are encoder-decoder models --
load both through `AutoModelForSeq2SeqLM`. Remember: **you do NOT train either model** in
this part, only run inference.

In [ ]:
# TODO (Section 2): Load both models + their tokenizers.
#
# Required variables when you're done:
#   MODEL_NAMES -- dict mapping a display label -> HF model id, for the two models
#   tokenizers  -- dict mapping the same labels -> AutoTokenizer instance
#   models      -- dict mapping the same labels -> AutoModelForSeq2SeqLM instance,
#                  moved to device and set to .eval()

MODEL_NAMES = {
    # "label 1": "google-t5/t5-small",
    # "label 2": "Falconsai/text_summarization",
}

tokenizers = {}
models = {}

raise NotImplementedError("TODO: load the two models + tokenizers (Section 2)")

## 3. Preprocess / tokenize the test set

Follow the assignment's IMPORTANT HINTS:
- Prefix each input with `"summarize: "`.
- Set `padding=True` in the tokenizer call (and `truncation=True`, `max_length=1024`).
- Convert the tokenized dataset to PyTorch tensors with `.set_format("torch")`.

Do this **separately for each model's tokenizer**, since the tokenized dataset must match
the tokenizer that will process it.

In [ ]:
# TODO (Section 3): Tokenize `news` once per model/tokenizer.
#
# Required variable when you're done:
#   tokenized -- dict mapping each label (same keys as MODEL_NAMES) -> a tokenized
#                Dataset with "input_ids" and "attention_mask" columns, in torch format

prefix = "summarize: "

tokenized = {}

raise NotImplementedError("TODO: tokenize the test set for each model (Section 3)")

## 4. Run inference: generate summaries for the whole test set

For each model, generate a summary for every example in the test set. You'll want to batch
this (e.g. with a `DataLoader`) rather than generating one example at a time -- see the HF
"Inference" section on Summarization linked in the assignment.

In [ ]:
# TODO (Section 4): Generate summaries with both models over the whole test set.
#
# Suggested helper:
#   def generate_summaries(model, tokenizer, tokenized_ds, batch_size=16, max_new_tokens=64):
#       ... batch the tokenized_ds, call model.generate(...), decode with
#           tokenizer.batch_decode(..., skip_special_tokens=True) ...
#       return predictions   # a list of strings, same length/order as news
#
# Required variables when you're done:
#   references  -- list of reference summaries (news["description"])
#   predictions -- dict mapping each label -> list of generated summary strings

references = None    # TODO
predictions = {}      # TODO

raise NotImplementedError("TODO: generate summaries for both models (Section 4)")

## 5. Metrics: ROUGE, Perplexity, BERTScore -- Requirement (*)

For each model, compute ROUGE (report rouge1, rouge2), Perplexity (average / 'mean_perplexity',
using gpt2), and BERTScore (precision, recall, F1). See the assignment for the `evaluate`
library usage examples -- **and read the "KNOWN ISSUE" note under the Perplexity example**:
the `evaluate` "perplexity" metric is currently broken on Colab's default package versions,
so you'll likely need the workaround shown there instead of calling it directly.

In [ ]:
# TODO (Section 5a): Set up the metric objects you'll need.
#
# rouge = load_metric("rouge")
# bertscore = load_metric("bertscore")
#
# For perplexity, see the assignment's "KNOWN ISSUE" note -- you'll likely need to load your
# own GPT-2 model/tokenizer directly (transformers.AutoModelForCausalLM) rather than relying
# on load_metric("perplexity", ...).

raise NotImplementedError("TODO: set up the ROUGE / Perplexity / BERTScore metrics (Section 5a)")

In [ ]:
# TODO (Section 5b): For each model, compute ROUGE, Perplexity, and BERTScore over the
# whole test set, and collect the results.
#
# Required variable when you're done:
#   metric_results -- dict mapping each label -> a dict with keys:
#                      "rouge1", "rouge2", "mean_perplexity",
#                      "bertscore_precision", "bertscore_recall", "bertscore_f1"
#
# Tip: guard against empty-string predictions (e.g. replace "" with ".") before scoring --
# some metrics error out on empty input.

metric_results = {}

raise NotImplementedError("TODO: compute ROUGE / Perplexity / BERTScore for each model (Section 5b)")

# pd.DataFrame(metric_results).T

## 6. Requirement (**): generate + compare summaries for two articles

Pick two articles from the test set. For each article, print the original text (or a
truncated preview), the reference summary, and each model's generated summary, so you can
compare them side by side in your write-up.

In [ ]:
# TODO (Section 6): Pick two example indices and print a side-by-side comparison.

example_idx = [0, 1]   # feel free to pick different indices

raise NotImplementedError("TODO: print the two-article qualitative comparison (Section 6)")

## 7. Save results (for the write-up)

Save your metrics table and the two qualitative examples to CSV (or however you'd like to
pull them into your write-up).

In [ ]:
# TODO (Section 7): Save metric_results and your two qualitative examples to disk.

raise NotImplementedError("TODO: save your results (Section 7)")